# MEN 4338 · Week 7 — Review mini-project: Copper South

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/iugb/men4338/notebooks/MEN4338_W7_review_copper_south.ipynb)

**IUGB · BSc Mining Engineering · Geostatistics (lecture & lab)** · Instructor: Laurent Kouadio

Half-way through the semester: time to run the whole workflow on a deposit you have never seen. Copper South
is a porphyry-style copper prospect (synthetic): 800 m × 600 m, drilled on a 60 m grid, plus one east–west
infill line at 30 m across the middle. The laboratory sent raw intervals.

You will go from the raw intervals to a validated variogram model, and summarise the result in a small table
as you would in a technical report. Every step is a skill from weeks 1 to 6.

**What you will practise:** compositing · declustering · declustered proportions · experimental variogram ·
model fitting · cross-validation · reporting.

Run each cell with **Shift + Enter**, from top to bottom, starting with the two setup cells.

In [ ]:
# Setup: run this cell first (Colab or local Jupyter).
# It fetches the data and the answer checks: from the repository (../data, ../tests)
# if you downloaded it, otherwise from GitHub.
import os, shutil, urllib.request
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/iugb/men4338/'
FILES = [('data/copper_south_intervals.csv', 'copper_south_intervals.csv'), ('data/copper_south_collars.csv', 'copper_south_collars.csv'), ('tests/w7/q1.py', 'tests/q1.py'), ('tests/w7/q2.py', 'tests/q2.py'), ('tests/w7/q3.py', 'tests/q3.py'), ('tests/w7/q4.py', 'tests/q4.py'), ('tests/w7/q5.py', 'tests/q5.py')]
os.makedirs('tests', exist_ok=True)
for source, target in FILES:
    if os.path.exists(target):
        continue
    local = os.path.join('..', source)
    if os.path.exists(local):
        shutil.copy(local, target)                      # offline kit / cloned repo
    else:
        urllib.request.urlretrieve(URL + source, target)  # Colab: download
print('Ready:', ', '.join(t for _, t in FILES))

In [ ]:
# Automatic answer checks (Otter-Grader)
try:
    import otter
except ImportError:
    %pip install -q otter-grader
    import otter
grader = otter.Notebook(tests_dir="tests")

In [ ]:
# Toolbox from earlier weeks: run this cell.
import numpy as np

def exp_variogram(x, y, v, lag, nlags, azimuth=None, atol=22.5):
    """Experimental semivariogram. Lag class k (1..nlags) holds the pairs with |d - k*lag| < lag/2.
    With an azimuth (degrees clockwise from north), only pairs within +/- atol of that direction are kept.
    Returns mean distance, semivariance and number of pairs of each class (NaN where a class is empty)."""
    x, y, v = (np.asarray(a, float) for a in (x, y, v))
    i, j = np.triu_indices(len(v), 1)
    dx, dy = x[j] - x[i], y[j] - y[i]
    d = np.hypot(dx, dy)
    keep = np.ones(len(d), bool)
    if azimuth is not None:
        az = np.degrees(np.arctan2(dx, dy)) % 180
        diff = np.abs(az - azimuth % 180)
        keep = np.minimum(diff, 180 - diff) <= atol
    sq = 0.5 * (v[j] - v[i]) ** 2
    hm, g, n = np.full(nlags, np.nan), np.full(nlags, np.nan), np.zeros(nlags, int)
    for k in range(1, nlags + 1):
        m = keep & (np.abs(d - k * lag) < lag / 2)
        n[k - 1] = m.sum()
        if n[k - 1]:
            hm[k - 1], g[k - 1] = d[m].mean(), sq[m].mean()
    return hm, g, n

def spherical(h, c0, c, a):
    """Spherical model: nugget c0, partial sill c, range a. gamma(0) = 0."""
    h = np.asarray(h, float)
    r = np.minimum(h / a, 1.0)
    return np.where(h > 0, c0 + c * (1.5 * r - 0.5 * r ** 3), 0.0)

def aniso_h(dx, dy, azimuth, a_major, a_minor):
    """Distance rescaled to the major range: an anisotropic model becomes model(aniso_h(...), c0, c, a_major)."""
    t = np.radians(azimuth)
    u = dx * np.sin(t) + dy * np.cos(t)
    w = dx * np.cos(t) - dy * np.sin(t)
    return np.sqrt(u ** 2 + (w * a_major / a_minor) ** 2)

def cell_weights(x, y, cell_size):
    """Cell declustering weights (sum to 1); cells start at the origin."""
    key = np.floor(np.asarray(x) / cell_size).astype(int) * 100000 + np.floor(np.asarray(y) / cell_size).astype(int)
    _, inv, cnt = np.unique(key, return_inverse=True, return_counts=True)
    w = 1.0 / cnt[inv]
    return w / w.sum()

def idw_estimate(x0, y0, x, y, v, power=2, nmax=12):
    """Inverse-distance estimate at (x0, y0) from the nmax closest samples."""
    d = np.hypot(x - x0, y - y0)
    k = np.argsort(d)[:nmax]
    if d[k[0]] < 1e-9:
        return float(v[k[0]])
    w = 1.0 / d[k] ** power
    return float((w * v[k]).sum() / w.sum())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

intervals = pd.read_csv("copper_south_intervals.csv")
collars = pd.read_csv("copper_south_collars.csv")
print(intervals.shape, collars.shape)
intervals.head()

**Reminders.** Composite = length-weighted mean of the assayed intervals. Declustered statistics use
`cell_weights(x, y, cell)`. A declustered **proportion** is the sum of the weights of the samples that satisfy
the condition. A lag is reliable with at least 30 pairs.

---
## Exercises

### Q1. Composites with coordinates

Build the DataFrame `cs` with one row per hole and the columns `hole_id`, `x`, `y`, `cu_pct` (the composite).
Then plot a location map coloured by grade.

In [ ]:
cs = ...
cs.head()

In [ ]:
grader.check("q1")

### Q2. Declustered statistics

Store the plain mean in `naive_cu`, the declustered mean with 60 m cells in `declus_cu`, and the declustered
proportion of composites above 0.5 % Cu in `prop_above_05`.

In [ ]:
x, y, cu = cs["x"].to_numpy(), cs["y"].to_numpy(), cs["cu_pct"].to_numpy()
naive_cu = ...
declus_cu = ...
prop_above_05 = ...
naive_cu, declus_cu, prop_above_05

In [ ]:
grader.check("q2")

### Q3. Experimental variogram

Compute the omnidirectional variogram of the composites with lag 60 m and 10 lags (`h_cu, g_cu, n_cu`) and
the sample variance `var_cu`. Plot it. Check the directions N000 and N090 quickly: is an isotropic model
acceptable here?

In [ ]:
h_cu, g_cu, n_cu = ...
var_cu = ...

In [ ]:
grader.check("q3")

### Q4. Fit a spherical model

Find `cu_params = (c0, c, a)` minimising the weighted SSE over $c_0$ = 0 to 0.06 (step 0.005), $c$ = 0.05 to
0.25 (step 0.01), $a$ = 60 to 600 m (step 20). Plot the model over the experimental points.

In [ ]:
cu_params = ...
cu_params

In [ ]:
grader.check("q4")

### Q5. Validate and report

Cross-validate IDW (power 2, 8 nearest composites) by leave-one-out: store the mean error in `cu_cv_me` and the
MSE in `cu_cv_mse`. Then fill the dictionary `summary` with the keys `declustered_mean`, `nugget`, `sill`
(total: $c_0 + c$), `range_m` and `cv_mse`, and print it as a small table.

In [ ]:
cu_cv_me = ...
cu_cv_mse = ...
summary = {
    "declustered_mean": ...,
    "nugget": ...,
    "sill": ...,
    "range_m": ...,
    "cv_mse": ...,
}
pd.Series(summary)

In [ ]:
grader.check("q5")

### Going further (not checked)

* Is the copper grade higher on the infill line than elsewhere? Was the line placed on a rich zone?
* Write the five numbers of `summary` in two sentences, as in the executive summary of a report.

---
**Before you leave:** run all the checks with the cell below, then save your notebook
(**File → Download → .ipynb** in Colab).

In [ ]:
grader.check_all()